# Моделирование: прогноз суточного PM2.5

**Вопрос.** Насколько точно можно предсказать среднесуточную концентрацию PM2.5 по вчерашнему значению, погоде и календарю, и лучше ли модели простого правила «завтра как сегодня»?

**Цель:** `pm25_mean` (µg/m³) из `data/processed/daily.csv`.

**Модели (L10–L11):**
- базовые ориентиры: **наивный прогноз** («завтра как сегодня» — `pm25_lag1`) и **среднее по месяцу** обучающей выборки (`month_mean`, ориентир для моделей без лага);
- **Linear Regression** (`StandardScaler` + `LinearRegression` в `Pipeline`) и **Random Forest** (`RandomForestRegressor`) — каждая в двух вариантах: со вчерашним `pm25_lag1` и без него («только погода и календарь»).

**Почему разбиение по времени, а не `train_test_split(shuffle=True)`:** соседние сутки сильно связаны по погоде и загрязнению. Случайное перемешивание перемешало бы прошлое и будущее: в обучении оказались бы дни рядом с тестовыми, что дало бы подглядывание в будущее и заниженную ошибку.

**Почему тест трогаем один раз:** если подбирать признаки, дату разбиения или гиперпараметры по метрикам теста, тестовая часть перестаёт быть независимой проверкой. Все решения принимаются только на `train` (через скользящую проверку по времени), а на `test` модели оцениваются один раз.

**Метрики:**
- `MAE` = среднее `|факт − прогноз|` (µg/m³, основная метрика);
- `RMSE` = √(среднее `(факт − прогноз)²`) — сильнее штрафует крупные промахи;
- `R²` = `1 − Σ(факт − прогноз)² / Σ(факт − среднее факта)²` — доля разброса, которую объясняет прогноз (может быть `< 0`).

**Оговорка о погоде:** в признаках используется фактическая суточная погода дня `D`, а не метеопрогноз. Поэтому результаты показывают «оценку сверху» для настоящего прогноза на завтра.

In [1]:
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sys.path.append("..")
from src import features, hypotheses, models

DAILY_PATH = "../data/processed/daily.csv"
FIGURES_DIR = "../figures"

# Целевая переменная: среднесуточный PM2.5 (µg/m³)
TARGET = "pm25_mean"

# Полный набор признаков (погода, календарные флаги и вчерашний PM2.5)
FEATURES = [
    "temp_c",
    "wind_speed",
    "humidity",
    "precip",
    "pressure",
    "heating_season",
    "is_weekend",
    "pm25_lag1",
]

# Набор без вчерашнего PM2.5: проверяем вклад только погоды и календаря
FEATURES_NO_LAG = [
    "temp_c",
    "wind_speed",
    "humidity",
    "precip",
    "pressure",
    "heating_season",
    "is_weekend",
]

# Граница разбиения: с 01.07.2024 начинается тестовый период с полным отопительным сезоном 2024/25
TEST_START = "2024-07-01"

# Число фолдов скользящей проверки по времени на train
N_SPLITS = 5

# Фиксируем зерно случайности для леса, bootstrap и перестановочной важности
SEED = 42
N_BOOT = 5000
N_REPEATS = 30

# Единая тема графиков: палитра, удобная для дальтоников
sns.set_theme(style="whitegrid", context="talk", palette="colorblind")

## Шаг 1. Таблица для модели

Обе модели (Linear Regression и Random Forest) учатся на одном наборе признаков, чтобы их сравнение было честным: метеопараметры дня (`temp_c`, `wind_speed`, `humidity`, `precip`, `pressure`), календарные флаги (`heating_season`, `is_weekend`) и вчерашний уровень загрязнения (`pm25_lag1`). Булевы флаги переводятся в `0.0` и `1.0`.

**Почему не берём часть колонок из `daily.csv`:**
- `month` (1…12) и `dayofweek` (0…6) как числа бессмысленны для линейной модели: декабрь (`12`) и январь (`1`) одинаково грязные, но стоят на разных концах шкалы, а коэффициент «на +1 месяц» нельзя объяснить. Сезонность уже несут `heating_season` и `temp_c`, а разницу будних и выходных дней — `is_weekend`.
- `is_calm` — это жёсткий порог от `wind_speed` (`< 1 м/с`), то есть дубль скорости ветра.
- `season` (текст) потребовал бы one-hot кодирования, тогда как сезонность уже учтена.

**Почему удаляем строки с пропусками, а не заполняем:** модели должны учиться и проверяться только на реальных измеренных сутках без выдуманных значений. Все шесть прогнозов (включая модели без лага) считаются на одних и тех же строках, чтобы сравнивать их на одинаковых сутках.

**Почему `pm25_lag1` не даёт утечки данных:** по построению в `features.lag_previous_day` это среднее PM2.5 за предыдущие календарные сутки (`D−1`), известное до начала дня `D`.

In [2]:
daily = features.read_daily_csv(DAILY_PATH)
features.check_daily(daily)

n_total = len(daily)
n_no_target = int(daily[TARGET].isna().sum())
n_no_lag_only = int((daily[TARGET].notna() & daily["pm25_lag1"].isna()).sum())

table = models.make_model_table(daily, FEATURES, TARGET)
assert len(table) == 1414, f"ожидалось 1414 строк, получено {len(table)}"

print(f"Всего суток в daily.csv: {n_total}")
print(f"Без цели (pm25_mean): {n_no_target}")
print(f"С целью, но без вчерашнего значения (pm25_lag1): {n_no_lag_only}")
print(f"Всего потеряно строк: {n_total - len(table)}")
print(f"Строк в таблице модели: {len(table)}")

Всего суток в daily.csv: 2044
Без цели (pm25_mean): 510
С целью, но без вчерашнего значения (pm25_lag1): 120
Всего потеряно строк: 630
Строк в таблице модели: 1414


## Шаг 2. Разбиение по времени

Разбиваем таблицу по границе `TEST_START = "2024-07-01"`: все сутки строго до неё идут в обучение (`train`), начиная с неё — в тест (`test`).

**Почему `2024-07-01`, а не `2025-07-01`:** данные станции заканчиваются `2025-11-13` (после `2025-11-14` станция не передаёт измерения). При границе `2025-07-01` в тесте остались бы только лето, осень и начало отопительного сезона без зимы — тогда модели главной проблемы (зимнего смога) не проверялись бы. Граница `2024-07-01` оставляет в `test` полный отопительный сезон `2024/25` вместе с летом и осенью `2025` года, а в `train` — данные `2020–2024` годов.

Решение о границе разбиения принято **до** расчёта каких-либо метрик на тесте.

In [3]:
train, test = models.split_by_date(table, TEST_START)

test_share = len(test) / len(table)
train_heating = int((train["heating_season"] == 1.0).sum())
test_heating = int((test["heating_season"] == 1.0).sum())

# Считаем, сколько календарных суток в тестовом периоде не попало в таблицу модели
test_calendar = daily.loc[daily["date"] >= pd.Timestamp(TEST_START), "date"]
test_missing_days = len(test_calendar) - len(test)

print(
    f"train: n = {len(train)} ({1 - test_share:.1%}), "
    f"{train['date'].min():%Y-%m-%d} ... {train['date'].max():%Y-%m-%d}, "
    f"отопительных суток: {train_heating}"
)
print(
    f"test:  n = {len(test)} ({test_share:.1%}), "
    f"{test['date'].min():%Y-%m-%d} ... {test['date'].max():%Y-%m-%d}, "
    f"отопительных суток: {test_heating}"
)
print(
    f"Календарных суток в периоде теста: {len(test_calendar)}, "
    f"пропущено в таблице модели: {test_missing_days}"
)

train: n = 931 (65.8%), 2020-04-11 ... 2024-06-30, отопительных суток: 504
test:  n = 483 (34.2%), 2024-07-01 ... 2025-11-13, отопительных суток: 213
Календарных суток в периоде теста: 501, пропущено в таблице модели: 18


**Что видим.** Из `2 044` суток в `daily.csv` потеряно `630` строк: `510` суток без цели `pm25_mean` и ещё `120` суток с целью, но без вчерашнего значения `pm25_lag1` (вывод шага 1). В итоговой таблице модели остаётся `1 414` суток с полным набором признаков и целью. После разбиения по границе `2024-07-01` обучающая часть `train` содержит `931` сутки (`65.8%`, `2020-04-11 … 2024-06-30`, из них `504` отопительных суток), а тестовая часть `test` — `483` суток (`34.2%`, `2024-07-01 … 2025-11-13`, из них `213` отопительных суток; вывод шага 2). Из `501` календарных суток тестового периода в таблицу модели не попали `18` суток из-за пропусков цели или лага (вывод шага 2).

## Шаг 3. Скользящая проверка по времени на train

Один контрольный отрезок даёт шумную оценку ошибки, а выбирать параметры модели по отложенной контрольной выборке нельзя — иначе итоговая проверка перестанет быть независимой. Поэтому качество и настройки моделей оцениваем только внутри `train` с помощью скользящей проверки по времени — `TimeSeriesSplit(n_splits=5)` ⚠️ (расширяющееся окно).

**Как устроен `TimeSeriesSplit`:** делим `train` по порядку времени на 6 последовательных кусков:
- фолд 1 — учимся на 1-м куске, проверяем на 2-м;
- фолд 2 — учимся на 1–2-м, проверяем на 3-м;
- и так далее до 5-го фолда — модель всегда учится на прошлом и проверяется на следующем куске.

Разбиение идёт по числу строк, поэтому из-за пропуска данных в 2022 году календарная длина кусков различается — ниже печатаем точные даты границ. В каждом фолде `mae_naive` — это ошибка наивного прогноза («завтра как сегодня», `pm25_lag1`) на том же проверочном куске.

In [4]:
# Проверяем линейную регрессию с лагом и без лага на фолдах train
cv_linreg = models.time_cv_table(train, FEATURES, models.make_linear_model(), N_SPLITS)
cv_linreg_no_lag = models.time_cv_table(
    train, FEATURES_NO_LAG, models.make_linear_model(), N_SPLITS
)

print("Linear Regression (с pm25_lag1) — скользящая проверка на train:")
print(cv_linreg.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

print("\nLinear Regression (без pm25_lag1) — скользящая проверка на train:")
print(cv_linreg_no_lag.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

Linear Regression (с pm25_lag1) — скользящая проверка на train:
 fold  train_end  val_start    val_end  n_train  n_val  mae_model  mae_naive
    1 2021-01-26 2021-02-04 2022-02-05      156    155     13.869     14.471
    2 2022-02-05 2022-02-06 2022-12-28      311    155     12.043     13.321
    3 2022-12-28 2022-12-29 2023-08-11      466    155      9.959      8.240
    4 2023-08-11 2023-08-12 2024-01-25      621    155     11.637     12.486
    5 2024-01-25 2024-01-26 2024-06-30      776    155      9.061      9.291

Linear Regression (без pm25_lag1) — скользящая проверка на train:
 fold  train_end  val_start    val_end  n_train  n_val  mae_model  mae_naive
    1 2021-01-26 2021-02-04 2022-02-05      156    155     24.306     14.471
    2 2022-02-05 2022-02-06 2022-12-28      311    155     16.750     13.321
    3 2022-12-28 2022-12-29 2023-08-11      466    155     15.717      8.240
    4 2023-08-11 2023-08-12 2024-01-25      621    155     14.597     12.486
    5 2024-01-25 2024-

## Шаг 4. Подбор Random Forest только на train

Для `RandomForestRegressor` подбираем три гиперпараметра на расширяющемся окне внутри `train`:
- `n_estimators` — число деревьев в лесу;
- `max_depth` — максимальная глубина каждого дерева;
- `min_samples_leaf` — минимальное число суток в листе дерева.

Перебираем сетку из 18 комбинаций (`{100, 300} × {4, 8, 16} × {1, 5, 20}`) отдельно для версии с лагом и без него. Лучшей считаем комбинацию с минимальным средним MAE по 5 фолдам (`cv_mae_mean`). Значение `None` для `max_depth` не используем: в `DataFrame` колонка с `None` и числами становится `float` с `NaN`, и лучшую строку нельзя передать обратно в модель как целые числа.

In [5]:
# Перебираем сетку гиперпараметров Random Forest только на обучающей выборке
FOREST_GRID = models.make_forest_grid([100, 300], [4, 8, 16], [1, 5, 20])

grid_lag = models.forest_grid_cv(train, FEATURES, FOREST_GRID, N_SPLITS, SEED)
grid_no_lag = models.forest_grid_cv(train, FEATURES_NO_LAG, FOREST_GRID, N_SPLITS, SEED)

print("Топ-5 сетки Random Forest (с pm25_lag1):")
print(grid_lag.head(5).to_string(index=False, float_format=lambda x: f"{x:.3f}"))

print("\nТоп-5 сетки Random Forest (без pm25_lag1):")
print(grid_no_lag.head(5).to_string(index=False, float_format=lambda x: f"{x:.3f}"))

# Извлекаем лучшие параметры как обычные int из первой строки отсортированной таблицы
best_lag = {
    "n_estimators": int(grid_lag.loc[0, "n_estimators"]),
    "max_depth": int(grid_lag.loc[0, "max_depth"]),
    "min_samples_leaf": int(grid_lag.loc[0, "min_samples_leaf"]),
}
best_no_lag = {
    "n_estimators": int(grid_no_lag.loc[0, "n_estimators"]),
    "max_depth": int(grid_no_lag.loc[0, "max_depth"]),
    "min_samples_leaf": int(grid_no_lag.loc[0, "min_samples_leaf"]),
}

print("\nВыбранные параметры (с лагом):", best_lag)
print("Выбранные параметры (без лага):", best_no_lag)

# Считаем ошибки по фолдам для лучших лесов
cv_forest = models.time_cv_table(
    train, FEATURES, models.make_forest(**best_lag, seed=SEED), N_SPLITS
)
cv_forest_no_lag = models.time_cv_table(
    train, FEATURES_NO_LAG, models.make_forest(**best_no_lag, seed=SEED), N_SPLITS
)

print("\nRandom Forest (с pm25_lag1) — фолды с лучшими параметрами:")
print(cv_forest.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

print("\nRandom Forest (без pm25_lag1) — фолды с лучшими параметрами:")
print(cv_forest_no_lag.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

# Сводим средний, минимальный и максимальный MAE по фолдам для всех вариантов
cv_sources = [
    ("naive", cv_linreg, "mae_naive"),
    ("linreg", cv_linreg, "mae_model"),
    ("forest", cv_forest, "mae_model"),
    ("linreg_no_lag", cv_linreg_no_lag, "mae_model"),
    ("forest_no_lag", cv_forest_no_lag, "mae_model"),
]
summary_rows = []
for name, frame, col in cv_sources:
    vals = frame[col]
    better_folds = int((frame[col] < frame["mae_naive"]).sum())
    summary_rows.append(
        {
            "model": name,
            "cv_mae_mean": float(vals.mean()),
            "cv_mae_min": float(vals.min()),
            "cv_mae_max": float(vals.max()),
            "folds_better_than_naive": better_folds,
        }
    )

cv_summary = pd.DataFrame(summary_rows)
print("\nСводка скользящей проверки на train (5 фолдов):")
print(cv_summary.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

Топ-5 сетки Random Forest (с pm25_lag1):
 n_estimators  max_depth  min_samples_leaf  cv_mae_mean  cv_mae_std
          300          8                 5        9.666       2.675
          300         16                 5        9.673       2.662
          300          8                 1        9.674       2.531
          300         16                 1        9.703       2.451
          300          4                 5        9.756       2.649

Топ-5 сетки Random Forest (без pm25_lag1):
 n_estimators  max_depth  min_samples_leaf  cv_mae_mean  cv_mae_std
          300          8                 5       12.326       3.719
          100          8                 5       12.328       3.730
          100         16                 5       12.332       3.720
          300         16                 5       12.336       3.705
          300          4                 5       12.356       3.706

Выбранные параметры (с лагом): {'n_estimators': 300, 'max_depth': 8, 'min_samples_leaf': 5}
Выбран


Random Forest (с pm25_lag1) — фолды с лучшими параметрами:
 fold  train_end  val_start    val_end  n_train  n_val  mae_model  mae_naive
    1 2021-01-26 2021-02-04 2022-02-05      156    155     12.952     14.471
    2 2022-02-05 2022-02-06 2022-12-28      311    155     11.170     13.321
    3 2022-12-28 2022-12-29 2023-08-11      466    155      6.506      8.240
    4 2023-08-11 2023-08-12 2024-01-25      621    155     10.312     12.486
    5 2024-01-25 2024-01-26 2024-06-30      776    155      7.392      9.291

Random Forest (без pm25_lag1) — фолды с лучшими параметрами:
 fold  train_end  val_start    val_end  n_train  n_val  mae_model  mae_naive
    1 2021-01-26 2021-02-04 2022-02-05      156    155     17.002     14.471
    2 2022-02-05 2022-02-06 2022-12-28      311    155     15.377     13.321
    3 2022-12-28 2022-12-29 2023-08-11      466    155      8.366      8.240
    4 2023-08-11 2023-08-12 2024-01-25      621    155     11.268     12.486
    5 2024-01-25 2024-01-26 202

**Что видим.** (вывод шагов 3–4)

- **Разброс по фолдам показывает «шум» одной проверки:** на 5 фолдах внутри `train` (по `155` суток проверки в каждом) ошибка сильно меняется от периода к периоду: у наивного прогноза `MAE` колеблется от `8.240` (фолд 3) до `14.471` µg/m³ (фолд 1), у `linreg` — от `9.061` до `13.869`, у `forest` — от `6.506` до `12.952`, у `linreg_no_lag` — от `12.837` до `24.306`, у `forest_no_lag` — от `8.366` до `17.002` µg/m³.
- **Выбранные параметры Random Forest:** для обеих версий (с `pm25_lag1` и без `pm25_lag1`) минимальный средний `MAE` на `train` дала одна и та же комбинация `n_estimators = 300`, `max_depth = 8`, `min_samples_leaf = 5` (`cv_mae_mean = 9.666` и `12.326` µg/m³ соответственно).
- **Сравнение с наивным прогнозом в CV:**
  - `forest` (среднее `9.666` против `11.562` µg/m³ у `naive`) точнее наивного во всех `5` из `5` фолдов;
  - `linreg` (среднее `11.314` µg/m³) точнее наивного в `4` из `5` фолдов (уступает на фолде 3: `9.959` против `8.240` µg/m³);
  - без вчерашнего `pm25_lag1` ошибка выше: `forest_no_lag` (среднее `12.326` µg/m³) точнее наивного только в `1` из `5` фолдов (фолд 4: `11.268` против `12.486` µg/m³), а `linreg_no_lag` (среднее `16.841` µg/m³) — ни в одном (`0` из `5` фолдов).